# 02 - Generate images (FLUX.1-dev)

Notebook form of `generate_images.py` - the headless script we actually ran on the GPU server
(DGX) with `nohup`. Both share the **exact same logic** (same fixed seeds, filenames, and manifest),
so this notebook reproduces the dataset.

It turns `data/prompts.json` (5 classes, attribute-based prompts) into the **undegraded** synthetic
image set in `data/synthetic_clean/{class}/`, plus a `manifest.csv`.

- **Model:** FLUX.1-dev (gated - needs a free HF account + accepted license + a read token).
- The output stays a **clean** photo; the CCTV degradation is added later in `03_degrade_and_augment`
  (keeps the with/without-degradation ablation valid).
- **Reproducible:** fixed `SEED` + a deterministic per-image seed. **Resumable:** skips images already on disk.

## 1. Configuration

These are the same knobs as `generate_images.py`'s CLI flags.

In [ ]:
# --- Configuration (mirror of generate_images.py's CLI flags) ---
SIZE = 512                 # image RESOLUTION in px (512 is ~4x faster than 1024; step 03 downscales anyway)
STEPS = 20                 # num_inference_steps
GUIDANCE = 3.5             # guidance_scale for FLUX-dev
IMAGES_PER_PROMPT = 1      # raise to 2-3 for a larger set (near-duplicate content per prompt)

PER_CLASS = 10             # cap prompts per class. 10 = quick pilot; set 0 for the FULL set
CLASSES = None             # e.g. ["full", "clean"]; None = all content classes
SKIP_UNCLASSIFIED = True   # 'unclassified' just borrows prompts; its degraded images are made in step 03
MODEL = "black-forest-labs/FLUX.1-dev"
OUT_DIR = None             # None -> a fresh versioned folder Diff_DataSet_vN; or set a string path

## 2. Imports

`utils.py` (next to this notebook) is the single source of truth - classes, seed, paths, prompts.

In [ ]:
import csv, sys, time
from pathlib import Path

# Make the local shlomi/utils.py importable whether launched from shlomi/ or the repo root.
for _cand in (Path.cwd(), Path.cwd() / "shlomi"):
    if (_cand / "utils.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand)); break
import utils

import torch
from diffusers import FluxPipeline

OUT_DIR = Path(OUT_DIR) if OUT_DIR else utils.next_versioned_dir()
MANIFEST = OUT_DIR / "manifest.csv"
print("classes :", utils.CLASS_NAMES)
print("seed    :", utils.SEED)
print("out dir :", OUT_DIR)

## 3. Hugging Face auth (one-time)

FLUX.1-dev is **gated**: accept its license on the model page, then authenticate. The token is never
hardcoded or committed - use a cached login or the `HF_TOKEN` env var.

In [ ]:
# Uncomment one of these the first time (token stays out of the notebook):
# from huggingface_hub import login; login(token="hf_xxx")   # paste your read token
# or run once in a terminal:  huggingface-cli login
# On the server we used a cached login / the HF_TOKEN environment variable.

## 4. Load prompts and the model

In [ ]:
prompts = utils.load_prompts()                 # reads data/prompts.json
print({c: len(v) for c, v in prompts.items()})

In [ ]:
print(f"loading {MODEL} (fp16 + cpu offload) - this takes a few minutes ...")
pipe = FluxPipeline.from_pretrained(MODEL, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()                # fits ~24GB FLUX: compute on GPU, idle modules in CPU RAM
pipe.set_progress_bar_config(disable=True)
print("loaded.")

## 5. Helpers (identical to generate_images.py)

In [ ]:
def seed_for(cls, p_i, k):
    """Deterministic per (class, prompt, image) so a pilot and a later full run agree."""
    return utils.SEED + utils.CLASS_TO_IDX[cls] * 1_000_000 + p_i * 100 + k

def plist(cls):
    """Prompts for a class, optionally capped by PER_CLASS."""
    return prompts[cls] if PER_CLASS <= 0 else prompts[cls][:PER_CLASS]

def generate(prompt, seed):
    gen = torch.Generator("cpu").manual_seed(seed)
    return pipe(prompt, height=SIZE, width=SIZE, guidance_scale=GUIDANCE,
                num_inference_steps=STEPS, max_sequence_length=512, generator=gen).images[0]

## 6. Generate

Resumable (skips images already on disk) and reproducible. Start with `PER_CLASS = 10` (a 40-image
pilot across the 4 content classes); once the pilot looks right, set `PER_CLASS = 0` and re-run for
the full set. Filenames are `{class}_{prompt_index:04d}_{k}.jpg`.

In [ ]:
classes = [c for c in utils.CLASS_NAMES
           if (CLASSES is None or c in CLASSES)
           and not (SKIP_UNCLASSIFIED and c == utils.UNCLASSIFIED)]
total = sum(len(plist(c)) for c in classes) * IMAGES_PER_PROMPT
print(f"target = {total} images @ {SIZE}px, {STEPS} steps | classes: {classes}")

t0 = time.time(); made = done = 0
for cls in classes:
    out_dir = utils.class_dir(OUT_DIR, cls)
    for p_i, prompt in enumerate(plist(cls)):
        for k in range(IMAGES_PER_PROMPT):
            done += 1
            fp = out_dir / f"{cls}_{p_i:04d}_{k}.jpg"
            if fp.exists():
                continue                       # resumable: skip what's already made
            generate(prompt, seed_for(cls, p_i, k)).save(fp, quality=95)
            made += 1
            rate = (time.time() - t0) / made
            print(f"{done}/{total}  [{cls}]  {rate:.0f}s/img  ETA ~{rate*(total-done)/60:.0f} min")
print(f"DONE. {made} new images this run | {(time.time()-t0)/60:.0f} min")

## 7. Manifest

Scans every class folder in `OUT_DIR` and records filepath / class / seed / model / prompt.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True); rows = 0
with open(MANIFEST, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["filepath", "class", "seed", "model", "prompt"])
    for cls in utils.CLASS_NAMES:
        cdir = OUT_DIR / cls
        if not cdir.is_dir():
            continue
        for fp in sorted(cdir.glob(f"{cls}_*.jpg")):
            p_i, k = (int(x) for x in fp.stem.split("_")[-2:])
            prompt = prompts[cls][p_i] if p_i < len(prompts[cls]) else ""
            w.writerow([str(fp), cls, seed_for(cls, p_i, k), MODEL, prompt]); rows += 1
print(f"manifest -> {MANIFEST}  ({rows} rows)")

## 8. Quick visual QA

A grid of a few images per class - confirm the four content states are visually distinct (especially `clean` vs `empty`) before scaling up.

In [ ]:
from PIL import Image, ImageDraw
n, sz = min(PER_CLASS or 8, 10), 200
shown = [c for c in utils.CLASS_NAMES if (OUT_DIR / c).is_dir() and any((OUT_DIR / c).glob("*.jpg"))]
grid = Image.new("RGB", (n * sz, len(shown) * sz), "white"); d = ImageDraw.Draw(grid)
for r, cls in enumerate(shown):
    for j, f in enumerate(sorted((OUT_DIR / cls).glob("*.jpg"))[:n]):
        grid.paste(Image.open(f).convert("RGB").resize((sz, sz)), (j * sz, r * sz))
    d.text((4, r * sz + 4), cls, fill="red")
grid     # displays inline in the notebook

## Full run on the server

For the full dataset we ran the **headless** script (survives disconnect, logs progress per image):

```bash
nohup python generate_images.py --per-class 0 --skip-unclassified > gen.log 2>&1 &
tail -f gen.log
```

It supports `--gpu`, `--classes`, `--per-class`, and `--out <folder>`. This notebook and
`generate_images.py` produce identical files (same seeds / names / manifest).